In [26]:
!pip install huggingface_hub

In [27]:
import os
import numpy as np
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("kaggle_hf_key")
api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")

Logged in as: equalNull


In [28]:
def build_query(file_path, global_columns, K=2**16):
    select_parts = []

    for col in global_columns:
        expr = f"""
        CASE
            WHEN TRY_CAST({col} AS DOUBLE) IS NOT NULL
                THEN CAST({col} AS DOUBLE)
            ELSE hash(CAST("{col}" AS VARCHAR)) % {K}
        END AS "{col}"
        """
        select_parts.append(expr)

    query = f"""
    SELECT {", ".join(select_parts)}
    FROM read_parquet('{file_path}', union_by_name=True)
    """
    return query

In [29]:
def process_file_duckdb(file_path, global_columns, con, batch_size=100_000):
    # Step 1: build query
    query = build_query(file_path, global_columns)

    # Step 2: execute query
    result = con.execute(query)

    # Step 3: fetch in batches
    while True:
        chunk = result.fetch_df_chunk(batch_size)
        if chunk is None or len(chunk) == 0:
            break

        yield chunk.to_numpy(dtype="float32")

In [30]:
global_columns = ['action', 'actorID', 'acuity_level', 'base_address', 'command_line', 
                  'context_info', 'data', 'dest_ip', 'dest_port', 'direction', 'end_time', 
                  'file_path', 'hostname', 'id', 'image_path', 'info_class', 'is_malicious', 
                  'key', 'l4protocol', 'logon_id', 'module_path', 'name', 'new_path', 'object', 
                  'objectID', 'parent_image_path', 'path', 'payload', 'pid', 'ppid', 'principal', 
                  'privileges', 'requesting_domain', 'requesting_logon_id', 'requesting_user', 
                  'service_type', 'sid', 'size', 'src_ip', 'src_pid', 'src_port', 'src_tid', 
                  'stack_base', 'stack_limit', 'start_address', 'start_time', 'start_type', 
                  'subprocess_tag', 'task_name', 'task_pid', 'task_process_uuid', 'tgt_pid', 
                  'tgt_pid_uuid', 'tgt_tid', 'tid', 'timestamp', 'type', 'user', 'user_name', 
                  'user_stack_base', 'user_stack_limit', 'value']

In [31]:
def process_folders(parquet_folder, global_columns, output_folder, con):
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(parquet_folder):
        if file.endswith(".parquet"):
            path = os.path.join(parquet_folder, file)

            print(f"Processing {file}")

            batch_idx = 0
            for batch in process_file_duckdb(path, global_columns, con):
                save_path = os.path.join(output_folder, f"{file}_{batch_idx}.npy")
                np.save(save_path, batch)
                batch_idx += 1

In [32]:
import duckdb, shutil
from huggingface_hub import snapshot_download

repo_id = f"equalNull/hosts100-labelled-optc"
all_columns = set()
working_path = "/kaggle/temp"
os.makedirs(working_path, exist_ok=True)
con = duckdb.connect(database=':memory:')
for day in range(16, 17):
    downloaded_path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=working_path,
        allow_patterns=f"sep{day}/*",
        cache_dir=working_path,
        max_workers=4
    )

    process_folders(
        parquet_folder=f"{working_path}/sep{day}/",
        global_columns=global_columns,
        output_folder="/kaggle/working/",
        con=con
    )
    con.close()
    shutil.rmtree(working_path)
    os.makedirs(working_path, exist_ok=True)

Fetching 93 files:   0%|          | 0/93 [00:00<?, ?it/s]

Processing AIA-301-325.ecar-2019-09-16-sysclient0319.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing AIA-151-175.ecar-2019-09-16-sysclient0175.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing AIA-351-375.ecar-2019-09-16-sysclient0357.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing AIA-301-325.ecar-2019-09-16-sysclient0301.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))